# Deployment of Endpoint with AWS Sagemaker and Inferencing

### Technologies we will learn/use in this projects
1. S3=buckets -- Boto3
2. Iam roles and Users
3. Complete Infrastructure of Sagemaker- Training and Endpoints

In [4]:
import sagemaker
from sklearn.model_selection import train_test_split
import pandas as pd 
import boto3
from sagemaker.core.helper.session_helper import Session

In [5]:
boto_session = boto3.Session(region_name="us-east-1")

sm_boto3 = boto_session.client("sagemaker")

sess = Session(boto_session=boto_session)

region = boto_session.region_name

bucket = "mobilesagemaker23"

print("Region:", region)
print("Using bucket:", bucket)

[03/12/26 00:43:59] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=672440;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\botocore\credentials.py\credentials.py]8;;\:]8;id=721946;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\botocore\credentials.py#1278\1278]8;;\

Region: us-east-1
Using bucket: mobilesagemaker23


In [6]:
print(region)

us-east-1


### Reading the CSV File and doing EDA

In [7]:
df=pd.read_csv("mob_price_classification_train.csv")
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [8]:
df.shape

(2000, 21)

In [9]:
df.isnull().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [10]:
df['price_range'].value_counts()    

price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64

In [11]:
features=list(df.columns)
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi',
 'price_range']

In [12]:
label=features.pop(-1)
label

'price_range'

In [13]:
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi']

In [14]:
X=df[features]
y=df[label]

In [15]:
Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.25,random_state=0)

In [16]:
print(Xtrain.shape)
print(Xtest.shape)

(1500, 20)
(500, 20)


In [17]:
trainX=pd.DataFrame(Xtrain)
trainX[label]=ytrain

testX=pd.DataFrame(Xtest)
testX[label]=ytest

In [18]:
trainX

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
1045,531,0,1.1,0,10,0,63,0.7,189,7,...,145,1903,2958,17,1,19,0,1,0,2
937,764,1,1.2,1,1,0,13,1.0,152,8,...,361,511,3148,18,7,6,1,1,0,2
1658,1812,1,1.3,1,4,1,42,1.0,162,7,...,380,1550,3338,18,13,11,1,1,1,3
529,1821,0,0.9,0,9,1,12,0.3,114,1,...,97,1803,2430,7,4,6,1,1,1,2
895,1790,1,2.3,1,3,1,49,0.5,100,3,...,396,1980,3568,6,2,18,1,0,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,1224,1,1.6,0,9,0,33,1.0,157,1,...,522,563,3796,10,5,13,1,1,0,3
1216,1158,0,0.7,1,1,1,29,0.7,123,2,...,311,1796,1542,17,9,15,1,0,1,1
1653,1190,0,2.0,1,0,0,40,0.2,93,5,...,1399,1646,3610,13,7,9,0,0,1,3
559,1191,0,2.4,1,2,0,13,0.9,169,1,...,179,1813,1028,14,6,8,1,1,1,0


In [19]:
trainX.to_csv("train-V-1.csv",index=False)
testX.to_csv("test-V-1.csv",index=False)

In [20]:
bucket

'mobilesagemaker23'

In [21]:
# Sending data to S3. SAGEMAKER WILL TAKE THE DATA FOR TRAINIG FROM S3
sk_prefix="sagemaker/mobile_price_classification/sklearncontainer"
trainpath=sess.upload_data(path="train-V-1.csv",bucket=bucket,key_prefix=sk_prefix)

testpath=sess.upload_data(path='test-V-1.csv',bucket=bucket,key_prefix=sk_prefix)

print(trainpath)
print(testpath)

s3://mobilesagemaker23/sagemaker/mobile_price_classification/sklearncontainer/train-V-1.csv
s3://mobilesagemaker23/sagemaker/mobile_price_classification/sklearncontainer/test-V-1.csv


## Script used by AWS Sagemaker to Train Model

In [31]:
%%writefile script.py


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import sklearn
import joblib
import os
import pandas as pd
import argparse


def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf


if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--random_state", type=int, default=0)

    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))

    parser.add_argument("--train-files", type=str, default="train-V-1.csv")
    parser.add_argument("--test-files", type=str, default="test-V-1.csv")

    args, _ = parser.parse_known_args()

    print("Sklearn Version:", sklearn.__version__)
    print("Joblib Version:", joblib.__version__)

    train_df = pd.read_csv(os.path.join(args.train, args.train_files))
    test_df = pd.read_csv(os.path.join(args.test, args.test_files))

    label = "price_range"
    features = list(train_df.columns)
    features.remove(label)

    X_train = train_df[features]
    y_train = train_df[label]

    X_test = test_df[features]
    y_test = test_df[label]

    print("Training Random Forest Model...")

    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        random_state=args.random_state,
        n_jobs=1
    )

    model.fit(X_train, y_train)

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)

    print("Model saved at:", model_path)

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

Overwriting script.py


### AWS Sagemaker Entry Point To Execute The Training script

In [32]:
from sagemaker.sklearn.estimator import SKLearn

print("SageMaker sklearn estimator working")

SageMaker sklearn estimator working


In [33]:
from sagemaker.sklearn.estimator import SKLearn

FRAMEWORK_VERSION = "1.2-1"

sklearn_estimator = SKLearn(
    entry_point="script.py",
    role="arn:aws:iam::642415824308:role/access_sagemaker",
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-Custom-Sklearn-Container",
    hyperparameters={
        "n_estimators":100,
        "random_state":0
    },
    use_spot_instances=True,
    max_wait=7200,
    max_run=3600
)


sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\sanju\AppData\Local\sagemaker\sagemaker\config.yaml


In [34]:
# launch training job, with asynchronous call
sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=True)

Using provided s3_resource


[03/12/26 01:02:23] INFO     Creating training-job with name:                                        ]8;id=309495;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py\session.py]8;;\:]8;id=437449;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py#951\951]8;;\
                             RF-Custom-Sklearn-Container-2026-03-11-19-32-18-036                                   

2026-03-11 19:32:26 Starting - Starting the training job...
2026-03-11 19:32:41 Starting - Preparing the instances for training...
2026-03-11 19:33:32 Downloading - Downloading the training image......
2026-03-11 19:34:38 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-11 19:34:38,913 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-03-11 19:34:38,918 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-11 19:34:38,922 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-11 19:34:38,939 sagemaker_sklea

### To get the model from S3

In [36]:
sklearn_estimator.latest_training_job.describe()
artifact=sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

In [37]:
artifact


's3://sagemaker-us-east-1-642415824308/RF-Custom-Sklearn-Container-2026-03-11-19-32-18-036/output/model.tar.gz'

## Deploying the Model for Endpoint

In [41]:
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime,strftime

model_name="Custom-sklearn-model-" +strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model=SKLearnModel(
    name=model_name,
    model_data=artifact,role="arn:aws:iam::642415824308:role/access_sagemaker",
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION
)

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\sanju\AppData\Local\sagemaker\sagemaker\config.yaml


In [42]:
model

In [43]:
## Endpoint Deployment

endpoint_name="Custom-sklearn-model-" +strftime("%Y-%m-%d-%H-%M-%S", gmtime()) 
print("EndpointName={}".format(endpoint_name))

predictor=model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name
)

EndpointName=Custom-sklearn-model-2026-03-11-19-45-29
sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\sanju\AppData\Local\sagemaker\sagemaker\config.yaml


[03/12/26 01:15:35] INFO     Creating model with name: Custom-sklearn-model-2026-03-11-19-45-22     ]8;id=683779;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py\session.py]8;;\:]8;id=215124;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py#3645\3645]8;;\

[03/12/26 01:15:37] INFO     Creating endpoint-config with name                                     ]8;id=703309;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py\session.py]8;;\:]8;id=194384;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py#5321\5321]8;;\
                             Custom-sklearn-model-2026-03-11-19-45-29                                              

                    INFO     Creating endpoint with name Custom-sklearn-model-2026-03-11-19-45-29   ]8;id=349388;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py\session.py]8;;\:]8;id=872849;file://c:\Data_Science\aws_sagemaker\nvenv\lib\site-packages\sagemaker\session.py#4223\4223]8;;\

-------!

In [44]:
predictor

In [47]:
testX[features][4:8]

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
1754,1086,1,1.7,1,0,1,43,0.2,111,6,1,56,1150,3285,11,5,17,1,1,0
1178,909,1,0.5,1,9,0,30,0.4,97,3,10,290,773,594,12,0,4,1,1,1
1533,642,1,0.5,0,0,1,38,0.8,86,5,10,887,1775,435,9,2,2,1,1,0
1303,888,0,2.6,1,2,1,33,0.4,198,2,17,327,1683,3407,12,1,20,1,0,0


In [49]:
print(predictor.predict(testX[features][4:8].values.tolist()))

[3 0 0 3]


In [50]:
sm_boto3.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': 'f7411a1b-02fa-43e6-9ebd-0204e901b3f1',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'f7411a1b-02fa-43e6-9ebd-0204e901b3f1',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Wed, 11 Mar 2026 19:52:56 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}